In [ ]:
# =====================================================
# PATCH-BASED TRAINING — SAFE UPGRADE (32 patches + Unet)
# =====================================================

!pip install -q segmentation-models-pytorch albumentations

import os, random, numpy as np, pandas as pd
import torch, rasterio, cv2
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from torch.optim.lr_scheduler import CosineAnnealingLR

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

BASE_PATH = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data"
image_dir = f"{BASE_PATH}/image"
mask_dir = f"{BASE_PATH}/label"
pred_dir = f"{BASE_PATH}/prediction/image"

train_ids = open(f"{BASE_PATH}/split/train.txt").read().splitlines()
val_ids = open(f"{BASE_PATH}/split/val.txt").read().splitlines()

# ── CLASS WEIGHTS ─────────────────────────────────────────────
counts = np.zeros(3, dtype=np.int64)
for id_ in train_ids:
    with rasterio.open(f"{mask_dir}/{id_}_label.tif") as src:
        lbl = src.read(1).astype(np.int64)
    for c in range(3):
        counts[c] += (lbl == c).sum()
freq = counts / counts.sum()
raw_weights = 1.0 / (freq + 1e-8)
raw_weights = raw_weights / raw_weights.sum()
raw_weights[1] *= 2.0
CLASS_WEIGHTS = raw_weights.tolist()

# ── PREPROCESSING ─────────────────────────────────────────────
def speckle_filter(hh, hv):
    hh = cv2.bilateralFilter(hh.astype(np.float32), d=5, sigmaColor=50, sigmaSpace=50)
    hv = cv2.bilateralFilter(hv.astype(np.float32), d=5, sigmaColor=50, sigmaSpace=50)
    return hh, hv

MEANS = np.array([841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551], dtype=np.float32)[:, None, None]
STDS  = np.array([473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894], dtype=np.float32)[:, None, None]

def load_image(img_path):
    with rasterio.open(img_path) as src:
        img = src.read().astype(np.float32)
    img[0], img[1] = speckle_filter(img[0], img[1])
    img = (img - MEANS) / STDS
    green, nir, swir = img[2], img[4], img[5]
    ndwi = np.clip((green - nir) / (np.abs(green + nir) + 1e-6), -3, 3)
    mndwi = np.clip((green - swir) / (np.abs(green + swir) + 1e-6), -3, 3)
    return np.concatenate([img, ndwi[None], mndwi[None]], axis=0)

# ── PRE-LOAD ──────────────────────────────────────────────────
print("Pre-loading images...")
image_cache = {}
mask_cache = {}
for id_ in train_ids + val_ids:
    image_cache[id_] = load_image(f"{image_dir}/{id_}_image.tif")
    with rasterio.open(f"{mask_dir}/{id_}_label.tif") as src:
        mask_cache[id_] = src.read(1).astype(np.int64)
print(f"✅ Cached {len(image_cache)} images")

# ── AUGMENTATIONS (milder than last version) ──────────────────
PATCH_SIZE = 256
train_transform = A.Compose([
    A.D4(p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.35),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.25),
    A.CoarseDropout(max_holes=6, max_height=28, max_width=28, fill_value=0, p=0.3),
])

# ── PATCH DATASET ─────────────────────────────────────────────
class PatchFloodDS(Dataset):
    def __init__(self, ids, image_cache, mask_cache, patch_size=256, patches_per_image=32, transform=None):
        self.ids = ids
        self.image_cache = image_cache
        self.mask_cache = mask_cache
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.transform = transform
        self.total = len(ids) * patches_per_image
        print(f"Dataset: {len(ids)} images × {patches_per_image} patches = {self.total} patches")

    def __len__(self): return self.total

    def __getitem__(self, idx):
        img_idx = idx // self.patches_per_image
        id_ = self.ids[img_idx]
        image = self.image_cache[id_]
        mask = self.mask_cache[id_]
        H, W = image.shape[1], image.shape[2]
        ps = self.patch_size
        top = random.randint(0, H - ps)
        left = random.randint(0, W - ps)
        image_patch = image[:, top:top+ps, left:left+ps]
        mask_patch = mask[top:top+ps, left:left+ps]
        if self.transform:
            aug = self.transform(image=image_patch.transpose(1,2,0), mask=mask_patch)
            image_patch = aug["image"].transpose(2,0,1)
            mask_patch = aug["mask"]
        return torch.tensor(image_patch, dtype=torch.float32), torch.tensor(mask_patch, dtype=torch.long)

class FullImageFloodDS(Dataset):
    def __init__(self, ids, image_cache, mask_cache):
        self.ids = ids
        self.image_cache = image_cache
        self.mask_cache = mask_cache
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        id_ = self.ids[idx]
        return (torch.tensor(self.image_cache[id_], dtype=torch.float32),
                torch.tensor(self.mask_cache[id_], dtype=torch.long))

# ── DATASETS ──────────────────────────────────────────────────
train_dataset = PatchFloodDS(train_ids, image_cache, mask_cache, patches_per_image=32, transform=train_transform)
val_dataset = FullImageFloodDS(val_ids, image_cache, mask_cache)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False, num_workers=2)

# ── MODEL (back to Unet) ──────────────────────────────────────
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=8,
    classes=3
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("✅ Unet + 32 patches per image")

# ── LOSS (BoundaryLoss at lower weight) ───────────────────────
cw = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(device)
ce_loss = torch.nn.CrossEntropyLoss(weight=cw)
dice_loss = smp.losses.DiceLoss(mode="multiclass")

class BoundaryLoss(torch.nn.Module):
    def __init__(self): super().__init__()
    def forward(self, logits, target):
        prob = torch.softmax(logits, dim=1)
        target_onehot = torch.nn.functional.one_hot(target, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()
        def sobel_edge(x):
            sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
            sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
            edge_x = torch.nn.functional.conv2d(x, sobel_x, padding=1)
            edge_y = torch.nn.functional.conv2d(x, sobel_y, padding=1)
            return torch.sqrt(edge_x**2 + edge_y**2 + 1e-6)
        pred_edge = sobel_edge(prob[:, 1:2, :, :])
        target_edge = sobel_edge(target_onehot[:, 1:2, :, :])
        return torch.mean(torch.abs(pred_edge - target_edge))

def loss_fn(p, t):
    return 0.4 * ce_loss(p, t) + 0.4 * dice_loss(p, t) + 0.1 * BoundaryLoss()(p, t)   # ← lower weight

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

# ── TRAINING ──────────────────────────────────────────────────
EPOCHS = 50
snapshot_paths = []
best_val_iou = 0.0
best_path = "/kaggle/working/best_patch_model.pth"

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        loss = loss_fn(model(images), masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        model.eval()
        ious = []
        with torch.no_grad():
            for images, masks in val_loader:
                probs = torch.softmax(model(images.to(device)), dim=1)[:,1].cpu().numpy()
                pred_m = (probs > 0.35).astype(np.uint8)
                true_m = (masks.numpy() == 1).astype(np.uint8)
                inter = np.logical_and(pred_m, true_m).sum(axis=(1,2))
                union = np.logical_or(pred_m, true_m).sum(axis=(1,2))
                ious.extend(np.where(union == 0, 1.0, inter / union))
        val_iou = np.mean(ious)
        print(f"Epoch {epoch+1:2d} | Loss:{total_loss/len(train_loader):.4f} | Val IoU:{val_iou:.4f}")
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save(model.state_dict(), best_path)
            print(f" ✅ Best saved! {val_iou:.4f}")

snapshot_paths = [best_path] + snapshot_paths
print(f"\nBest Val IoU: {best_val_iou:.4f}")

In [ ]:
# ========================== THRESHOLD TUNING ==========================
@torch.no_grad()
def compute_flood_iou(threshold):
    model.eval()
    ious = []
    for images, masks in val_loader:
        images = images.to(device)
        preds = model(images)
        prob = torch.softmax(preds, dim=1)[:, 1].cpu().numpy()
        pred_mask = (prob > threshold).astype(np.uint8)
        true_mask = (masks.numpy() == 1).astype(np.uint8)
        intersection = np.logical_and(pred_mask, true_mask).sum(axis=(1,2))
        union = np.logical_or(pred_mask, true_mask).sum(axis=(1,2))
        iou = np.where(union == 0, 1.0, intersection / union)
        ious.extend(iou)
    return np.mean(ious)

print("Tuning flood threshold on validation set...")
best_thresh, best_iou = 0.45, 0.0
for thresh in np.arange(0.30, 0.71, 0.01):
    iou = compute_flood_iou(thresh)
    if iou > best_iou:
        best_iou, best_thresh = iou, thresh
print(f"✅ Optimal flood threshold: {best_thresh:.3f} (val IoU = {best_iou:.4f})")

# ========================== MULTI-SCALE + 8-WAY TTA ==========================
@torch.no_grad()
def predict_tta_multi_scale(model, img: torch.Tensor) -> torch.Tensor:
    scales = [0.8, 1.0, 1.2]
    all_probs = []
    def _p(x): return torch.softmax(model(x), dim=1)

    def pad_to_multiple(x, multiple=16):
        h, w = x.shape[2], x.shape[3]
        new_h = ((h + multiple - 1) // multiple) * multiple
        new_w = ((w + multiple - 1) // multiple) * multiple
        pad_h = new_h - h
        pad_w = new_w - w
        padded = torch.nn.functional.pad(x, (0, pad_w, 0, pad_h), mode='constant', value=0)
        return padded, (pad_h, pad_w)

    for scale in scales:
        if scale != 1.0:
            orig_h, orig_w = img.shape[2], img.shape[3]
            new_h, new_w = int(orig_h * scale), int(orig_w * scale)
            scaled = torch.nn.functional.interpolate(img, size=(new_h, new_w), mode='bilinear', align_corners=False)
        else:
            scaled = img
            orig_h, orig_w = img.shape[2], img.shape[3]

        padded, (pad_h, pad_w) = pad_to_multiple(scaled)

        p0 = _p(padded)
        p1 = torch.flip(_p(torch.flip(padded, [3])), [3])
        p2 = torch.flip(_p(torch.flip(padded, [2])), [2])
        p3 = torch.rot90(_p(torch.rot90(padded, 1, [2,3])), -1, [2,3])
        p4 = torch.rot90(_p(torch.rot90(padded, 2, [2,3])), -2, [2,3])
        p5 = torch.rot90(_p(torch.rot90(padded, 3, [2,3])), -3, [2,3])
        img_h = torch.flip(padded, [3])
        p6 = torch.flip(torch.rot90(_p(torch.rot90(img_h, 1, [2,3])), -1, [2,3]), [3])
        p7 = torch.flip(torch.rot90(_p(torch.rot90(img_h, 3, [2,3])), -3, [2,3]), [3])

        avg = (p0 + p1 + p2 + p3 + p4 + p5 + p6 + p7) / 8.0

        if pad_h > 0 or pad_w > 0:
            avg = avg[:, :, :avg.shape[2]-pad_h, :avg.shape[3]-pad_w]

        if scale != 1.0:
            avg = torch.nn.functional.interpolate(avg, size=(orig_h, orig_w), mode='bilinear', align_corners=False)

        all_probs.append(avg)

    return torch.mean(torch.stack(all_probs), dim=0).squeeze(0)

def postprocess_flood(flood_mask: np.ndarray) -> np.ndarray:
    if flood_mask.max() == 0:
        return flood_mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    closed = cv2.morphologyEx(flood_mask, cv2.MORPH_CLOSE, kernel)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    clean = np.zeros_like(closed)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= 50:
            clean[labels == i] = 1
    return clean

def rle_encode(mask: np.ndarray) -> str:
    if mask.max() == 0:
        return "0 0"
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str, runs))

# ========================== FINAL INFERENCE ==========================
model.eval()
pred_ids = sorted([f.replace("_image.tif", "") for f in os.listdir(pred_dir)])
predictions = []

print(f"Running multi-scale 8-way TTA + ensemble on {len(pred_ids)} test images...")

for id_ in pred_ids:
    img_path = f"{pred_dir}/{id_}_image.tif"
    with rasterio.open(img_path) as src:
        image6 = src.read().astype(np.float32)
    image6 = (image6 - MEANS) / STDS
    green, nir, swir = image6[2], image6[4], image6[5]
    ndwi  = np.clip((green - nir)  / (np.abs(green + nir)  + 1e-6), -3, 3)
    mndwi = np.clip((green - swir) / (np.abs(green + swir) + 1e-6), -3, 3)

    image = np.concatenate([image6, ndwi[np.newaxis], mndwi[np.newaxis]], axis=0)
    image_t = torch.tensor(image, dtype=torch.float32).unsqueeze(0).to(device)

    ensemble_prob = None
    for snap_path in snapshot_paths:
        model.load_state_dict(torch.load(snap_path, map_location=device))
        prob = predict_tta_multi_scale(model, image_t)
        ensemble_prob = prob if ensemble_prob is None else ensemble_prob + prob
    ensemble_prob /= len(snapshot_paths)

    flood_prob = ensemble_prob[1].cpu().numpy()
    mask = (flood_prob > best_thresh).astype(np.uint8)
    mask = postprocess_flood(mask)

    rle = rle_encode(mask)
    predictions.append((id_, rle))

df = pd.DataFrame(predictions, columns=["id", "rle_mask"])
df.to_csv("submission_phase2.csv", index=False)

print(df.head())
print(f"\n✅ Submission saved: submission_phase2.csv ({len(df)} rows)")
print(f"Used threshold: {best_thresh:.3f} | Snapshots: {len(snapshot_paths)} | Multi-scale TTA enabled")